# 0.43 — Schema minimale per investment themes

**Versione proposta: 0.3.** Un vocabolario piccolo per annotare headline e costruire un grafo temporale:
**7 tipi di entità + EVENT, 8 driver economici, 7 relazioni economiche**.
Due collegamenti strutturali, `ACTOR` e `TARGET`, collegano gli eventi ai partecipanti.

La domanda è: *quali cambiamenti economici collegano più aziende agli stessi prodotti, concetti o mercati?*
Un tema sarà un sottografo emergente; il suo nome non viene fornito all'estrattore.

Questo notebook definisce schema, confini e formato delle annotazioni. Le celle sono eseguibili offline;
gli esempi sono sintetici e non richiedono modelli, API o dati di mercato.
La minimalità è una scelta operativa per un primo esperimento, non un risultato dimostrato dai paper.

## Come leggere lo schema: nodi, archi e attributi

Lo schema 0.3 definisce un **grafo eterogeneo**: contiene **8 tipi di nodo** e **9 tipi di relazione**
(7 economiche + 2 strutturali). Le relazioni sono direzionali, salvo `PARTNERS_WITH`, che è simmetrica.
Gli otto driver sono valori di un attributo degli eventi.

Un nodo è una cosa specifica, il suo tag ne indica il tipo; un arco collega due nodi.

| Tipo di nodo | Esempio illustrativo | Significato |
|---|---|---|
| `COMPANY` | Alba | Azienda |
| `ORGANIZATION` | Autorità di regolamentazione | Istituzione non aziendale |
| `PRODUCT` | Batterie allo stato solido | Prodotto o servizio |
| `CONCEPT` | Riciclo delle batterie | Tecnologia, processo o concetto |
| `SECTOR` | Industria automobilistica | Settore o mercato |
| `LOCATION` | Italia | Geografia |
| `INDICATOR` | Capacità produttiva di Alba | Variabile misurabile |
| `EVENT` | Il piano di Alba di aumentare la capacità | Fatto o cambiamento specifico |

**Alba è un nodo; COMPANY è il suo tipo.** Il nodo evento identifica un fatto particolare:
due piani di espansione di aziende diverse sono due eventi diversi.

| Relazione economica | Collegamento illustrativo | Informazione conservata |
|---|---|---|
| `ACTIVE_IN` | Alba → settore batterie | Attività o esposizione esplicita nel testo |
| `OFFERS` | Alba → batterie | Produzione, erogazione o introduzione di un'offerta |
| `INVESTS_IN` | Alba → tecnologia di riciclo | Investimento; verso aziende, partecipazione senza controllo |
| `ACQUIRES` | Alba → Borea | Acquisizione o proposta di acquisizione, con la relativa modalità |
| `PARTNERS_WITH` | Alba ↔ Borea | Collaborazione; l'ordine non conta |
| `SUPPLIES_TO` | Borea → Alba | Fornitore → cliente |
| `IMPACTS` | Una restrizione → settore batterie | Effetto attribuito esplicitamente dalla fonte |

I due archi strutturali sono **EVENT —ACTOR→ entità** e **EVENT —TARGET→ entità**:
descrivono i partecipanti di un evento. Un arco economico, invece, descrive una relazione specifica
come una fornitura o un investimento. Le firme complete dei tipi ammessi sono definite nelle celle successive.

### Un evento è un nodo: esempio completo

Headline sintetica: **“Alba pianifica un aumento del 20% della capacità produttiva di batterie.”**

Il grafo previsto dalla versione 0.3 è:

```text
              ┌── ACTOR ───→ Alba [COMPANY]
E1 [EVENT] ───┼── TARGET ──→ Batterie [PRODUCT]
              └── TARGET ──→ Capacità produttiva di Alba [INDICATOR]

Attributi di E1:
  description = piano di aumento della capacità del 20%
  driver      = SUPPLY
  modality    = planned
  event_time  = null  (la headline non indica quando avverrà)
```

- `ACTOR`: chi compie o pianifica l'azione; qui Alba.
- `TARGET`: l'oggetto, la variabile o il destinatario dell'evento; qui batterie e capacità produttiva.
- `driver`: la dimensione economica dell'evento; qui l'offerta.
- `modality`: lo stato dell'affermazione; qui un piano, non un aumento già completato.

`ACTOR` e `TARGET` sono **nomi degli archi**, non nodi. `SUPPLY` e `planned` sono **attributi di E1**,
non altri nodi o archi. La fonte è conservata tramite `source_id` e `evidence`; nello schema iniziale
non è richiesto un nodo documento.

In questo esempio `claims=[]`: l'aumento è rappresentato dall'evento, perché dalla versione 0.2
non esiste un predicato economico `INCREASES`. Alba e batterie sono comunque collegate attraverso E1.

Una fornitura può invece avere sia un arco economico sia un evento:

```text
Borea —SUPPLIES_TO→ Alba

E2 [EVENT] —ACTOR→ Borea
E2 [EVENT] —TARGET→ Alba
E2 [EVENT] —TARGET→ Sistemi di raffreddamento
```

L'arco diretto conserva chi fornisce a chi; l'evento conserva quella specifica fornitura,
i partecipanti, il tempo, la modalità e l'evidenza. Se rappresentano lo stesso fatto, non contano
come due prove indipendenti.

Per scoprire temi, eventi diversi possono collegarsi allo stesso prodotto o concetto:

```text
Alba  ←ACTOR— Espansione A —TARGET→ Batterie
Borea ←ACTOR— Espansione B —TARGET→ Batterie
```

Questi percorsi mostrano la struttura dei collegamenti: le frecce ACTOR non significano causalità.
Per unire menzioni dello stesso prodotto o della stessa azienda tra documenti serve entity linking;
gli ID locali del notebook, da soli, non realizzano ancora questo collegamento.

### Che cos'è il driver?

Il campo **driver è la categoria economica principale con cui classifichiamo un evento**.
Risponde alla domanda: *quale dimensione dell'attività economica riguarda questo fatto?*

Qui usiamo il termine di FinKario come etichetta di classificazione. **Non è una dimostrazione della
causa dell'evento**, un punteggio di importanza, una previsione di rendimento o il nome del tema investibile.

| Driver | Dimensione economica | Esempio sintetico |
|---|---|---|
| `SUPPLY` | Offerta, capacità, disponibilità di input | Alba pianifica più capacità produttiva |
| `DEMAND` | Domanda, ordini, consumi | Aumentano gli ordini di batterie |
| `REVENUE` | Ricavi, utili e risultati | Crescono i ricavi di Alba |
| `EFFICIENCY_COST` | Costi, efficienza, produttività | L'automazione riduce i costi di produzione |
| `STRATEGIC_ACTION` | Scelte strategiche e societarie | Alba acquisisce Borea |
| `TECHNOLOGY_INNOVATION` | Innovazione e nuovi prodotti | Alba presenta una nuova tecnologia di riciclo |
| `POLICY_REGULATION` | Regole, autorizzazioni, divieti | Un'autorità vieta un prodotto |
| `MACRO` | Tassi, ciclo e shock macroeconomici | Una banca centrale aumenta i tassi |

La differenza tra driver e causa è visibile nell'esempio: **SUPPLY** dice che il piano di espansione
riguarda l'offerta. La headline non dice *perché* Alba espanda la capacità; non possiamo completare
la causa con “domanda in aumento”.

Le categorie possono sovrapporsi. Un investimento in automazione può essere `STRATEGIC_ACTION`
se il fatto centrale è l'investimento, oppure `EFFICIENCY_COST` se il testo descrive una riduzione dei costi.
Annotiamo il driver principale supportato dal testo; se non è determinabile usiamo `null`.
Non assegniamo tutti i driver plausibili per conoscenza generale.

Un tema come l'espansione delle infrastrutture per data center può includere eventi di domanda,
offerta, innovazione e strategia. **Il tema emerge dai collegamenti tra eventi ed entità;
il driver aiuta a leggere il meccanismo economico dei singoli eventi.**

### Quanto è vicino ai paper?

- **Entità:** derivano soprattutto da [FinDKG, tabella 2, p. 4](../../docs/papers/2407.10909v2.pdf).
  Abbiamo rinominato o accorpato alcuni tipi; gli indicatori aziendali riprendono anche FinKario.
- **Relazioni economiche:** adattiamo la tabella 1 di FinDKG. `ACTIVE_IN` estende il suo
  `Operate In` geografico; `OFFERS` accorpa produzione e introduzione; `IMPACTS` accorpa i tre
  predicati di impatto. `ACQUIRES` e `PARTNERS_WITH` vengono dalla guideline di 0.42.
- **Driver:** le otto categorie riprendono [FinKario, A.2.3, p. 13](../../docs/papers/2026.acl-long.446.pdf).
  Gli esempi e i confini operativi sono il nostro adattamento alle headline.
- **Evento come nodo centrale:** è una nostra scelta di rappresentazione. La figura 10 di FinKario
  (p. 17) conserva descrizioni di evento/effetto, entità, timeframe e reasoning; non definisce
  esattamente i nostri identificatori e gli archi `ACTOR` e `TARGET`.

Lo schema è quindi un adattamento motivato dei due lavori. Il notebook definisce annotazioni e
vincoli; non costruisce ancora il grafo globale né dimostra la scoperta automatica di temi.

**Precisazioni della versione 0.2, recepite nelle definizioni della 0.3:**

1. In `IMPACTS`, la parola “dopo” da sola esprime successione temporale: non basta a stabilire un effetto.
   Nella 0.3 vale lo stesso per “as”, così la regola resta meccanica: servono un marcatore causale o un verbo di effetto.
2. `ACTIVE_IN` va definito semanticamente, indipendentemente dalla posizione delle parole;
   quando la frase afferma un singolo lancio o una produzione vale `OFFERS`, senza duplicarlo in `ACTIVE_IN`.
3. `owner_id` è un campo, non un arco di grafo già definito. Per “I ricavi di Alba aumentano”,
   l'evento ha TARGET i ricavi, con `owner_id` = Alba; Alba entra in ACTOR/TARGET solo se il testo
   la descrive come partecipante (scelta della 0.3). Gli indicatori condivisi, come tassi e prezzi degli input, possono collegare
   più aziende: la scelta di rappresentare le variazioni come eventi è una semplificazione,
   non una conseguenza dell'impossibilità degli indicatori di collegare aziende.

### Giustificazione delle scelte: un'ontologia adattata al compito

**Adattiamo l'ontologia alla scoperta di temi emergenti attraverso esposizioni economiche condivise.**
Ogni modifica ha una motivazione legata al compito e un compromesso da dichiarare.
La motivazione è una scelta progettuale; la superiorità rispetto agli schemi originali è un'ipotesi
da verificare, non un risultato già ottenuto.

Il nostro obiettivo e i nostri documenti differiscono dagli esperimenti dei paper.
[FinDKG, §5.3, p. 7](../../docs/papers/2407.10909v2.pdf) cerca esposizioni aziendali a un tema AI già selezionato.
[FinKario, §§3–4](../../docs/papers/2026.acl-long.446.pdf) organizza report per supportare previsioni su singole aziende.
Qui cerchiamo prodotti, tecnologie e cambiamenti che iniziano a collegare aziende diverse, usando headline
che contengono meno informazioni di articoli e report completi. Questa differenza motiva una selezione
e un adattamento delle categorie originali.

| Modifica | Motivazione per il nostro compito | Compromesso o limite |
|---|---|---|
| Riduzione dei tipi di entità | Conservare aziende, attività, concetti, contesto ed eventi necessari a riconoscere esposizioni condivise | Minore dettaglio istituzionale; persone e strumenti finanziari restano fuori dal primo perimetro |
| `ACTIVE_IN` estende `Operate In` | Collegare l'azienda anche a settori, prodotti e tecnologie associati esplicitamente alla sua attività | Associazione generale: non misura ricavi, intensità dell'esposizione o rilevanza economica |
| `OFFERS` accorpa `Produce` e `Introduce` | Riconoscere chi porta sul mercato una determinata offerta nel primo insieme di aziende esposte | Produzione consolidata e lancio sono diversi; la distinzione rimane nell'evento e nell'evidenza e non è più un predicato dedicato |
| `IMPACTS` accorpa i tre impatti | Separare il tipo di relazione dal suo segno, espresso come attributo | Serve un effetto attribuito dalla fonte; successione temporale e co-menzione non bastano |
| `ACQUIRES` e `PARTNERS_WITH` | Rappresentare operazioni che collegano aziende e competenze anche prima di una classificazione tematica stabile | Un accordo non dimostra esposizione materiale o beneficio economico |
| `SUPPLIES_TO` | Attraversare la catena del valore e trovare aziende coinvolte indirettamente nello stesso cambiamento | Una fornitura non rivela da sola dipendenza, dimensione del contratto o quota di ricavi |
| Variazioni quantitative rappresentate come eventi | Conservare aumenti o riduzioni di capacità, domanda e risultati anche quando è nominata una sola azienda | La direzione della variazione deve restare recuperabile |
| EVENT con `ACTOR` e `TARGET` | Conservare insieme partecipanti, oggetto, tempo, modalità ed evidenza di un fatto | Più nodi; occorre riconoscere repliche della stessa notizia o dello stesso evento |
| Otto driver economici | Classificare la dimensione economica dei cambiamenti con un vocabolario generale già presente in FinKario | Categorie sovrapposte: driver principale o null; nessuna causalità o investibilità automatica |

Le corrispondenze di partenza sono nelle tabelle 1–2 di FinDKG (p. 4) e nell'appendice A.2.3 di FinKario
(p. 13). Le regole azienda–azienda riprese da 0.42 sono documentate nella
[linea guida locale](gold/guideline.txt). L'uso sistematico di eventi con ACTOR e TARGET è una nostra scelta.

Il disegno separa quattro domande:

- **Tipo di entità:** di che cosa stiamo parlando?
- **Relazione:** quale collegamento viene affermato?
- **Evento:** che cosa succede, quando e a chi?
- **Driver:** quale dimensione economica riguarda il cambiamento?

Questa separazione evita di usare azienda, acquisizione, domanda e intelligenza artificiale come categorie
dello stesso livello. I tipi generali sono fissati, mentre nuovi prodotti e concetti possono entrare
senza un elenco preventivo di temi.

**Informazione conservata e informazione persa.** L'accorpamento non è automaticamente senza perdita:
la citazione originale rende possibile recuperare dettagli, ma non li rende già disponibili come campi
strutturati. La rimozione di Raise e Decrease è giustificata dallo spostamento della variazione nell'evento,
non dall'impossibilità degli indicatori di collegare aziende. Tassi, prezzi degli input e domanda possono
riguardarne molte. Un possibile completamento è `change_direction` sull'evento quando esplicito:
**è una proposta, non un attributo implementato nella versione 0.3**.

### Quali ipotesi dobbiamo verificare?

Un vocabolario più piccolo e confini chiari potrebbero facilitare l'annotazione delle headline.
Meno categorie, però, non significano automaticamente maggiore accuratezza.
La valutazione proposta separa fedeltà alla fonte, copertura informativa e utilità per la discovery.

1. **Confronto sullo stesso corpus.** Annotare lo stesso campione con lo schema originale di FinDKG
   e con quello adattato. Esplicitare le corrispondenze tra etichette: un accorpamento rende alcune
   distinzioni più facili e non va contato, da solo, come miglioramento dell'estrazione.
2. **Qualità delle annotazioni.** Misurare precisione, richiamo e accordo tra annotatori indipendenti
   per entità, direzione delle relazioni, modalità ed evidenza. Per eventi e driver, valutare i relativi
   campi senza attribuire allo schema originale output che non prevede.
3. **Copertura e perdita di informazione.** Registrare fatti utili conservati, omessi o forzati in
   categorie troppo generiche. Verificare se OFFERS e ACTIVE_IN introducono ambiguità e se la direzione
   delle variazioni resta utilizzabile.
4. **Contributo delle scelte.** Confrontare relazioni dirette da sole, relazioni più eventi, e la stessa
   rappresentazione con i driver. I driver non cambiano da soli la topologia: questo confronto è utile
   se il metodo di discovery ne utilizza i valori.
5. **Utilità temporale.** Valutare i temi individuati su periodi successivi, mantenendo fissi definizioni,
   soglie e protocollo; usare fonti disponibili alla data e un campione di test nuovo, dato che le v0.2 e v0.3
   è stata progettata anche osservando il test di 0.42.

Si tratta di un piano di valutazione: questo notebook non contiene ancora risultati di tali confronti.
La qualità dell'estrazione, quella dei temi e la performance di un eventuale portafoglio sono risultati
distinti e non vanno sostituiti l'uno all'altro.

**Formulazione per la metodologia del lavoro:**

> Adottiamo uno schema orientato alla scoperta di temi emergenti, derivato da FinDKG e FinKario.
> Le modifiche comprendono accorpamenti semantici, estensioni per rappresentare esposizioni economiche
> e una rappresentazione esplicita degli eventi con partecipanti e metadati temporali. Queste scelte
> mirano a rendere annotabili le informazioni disponibili nelle headline e a collegare aziende
> attraverso prodotti, tecnologie e cambiamenti condivisi. Il protocollo proposto valuta separatamente
> la fedeltà alle fonti, la copertura informativa e l'utilità delle modifiche per la scoperta di temi.

## 1. Fondamento nella letteratura

| Fonte locale | Passaggio da usare | Cosa riprendiamo |
|---|---|---|
| **[F1] Li & Sanna Passino (2024), FinDKG** — [PDF](../../docs/papers/2407.10909v2.pdf) | Tabelle 1–2, p. 4 PDF; §3.1, p. 3 | Tipi di entità, relazioni direzionali e data della fonte |
| **[F2] Li et al. (2026), FinKario** — [PDF](../../docs/papers/2026.acl-long.446.pdf) | §3.1, pp. 3–4; A.2.3, p. 13 PDF / 9838; figura 10, p. 17 / 9842 | Separazione attributi/eventi, otto driver, partecipanti, timeframe e testo sorgente |

[F1] parte da **12 tipi di entità e 15 predicati**. [F2] descrive **11 relazioni di attributo e 8 categorie
di driver**: queste ultime diventano qui tag degli eventi, non otto predicati fra aziende.
Le tabelle seguenti dichiarano quali elementi sono rinominati, accorpati o adattati.

**Limiti del trasferimento.** I paper lavorano su articoli e report completi: con sole headline avremo meno
informazione e più campi vuoti. FinDKG §5.3 (p. 7) cerca esposizioni a un tema AI già scelto;
FinKario valuta previsioni settimanali su azioni. La scoperta aperta di temi richiede una valutazione
aggiuntiva. Nessuno dei due fornisce, da solo, una garanzia completa di assenza di leakage.

**Criterio di inclusione.** Manteniamo ciò che serve a collegare azienda → attività/concetto →
cambiamento economico, più contesto geografico e istituzionale. Prezzi, ticker, rating, azionisti e persone
restano fuori dal primo schema di estrazione; potranno essere collegati successivamente da fonti storiche.

In [1]:
import json
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)
SCHEMA_VERSION = "0.3"

## 2. Tag delle entità

I nomi delle entità sono aperti: una tecnologia sconosciuta può entrare come `CONCEPT`.
`EVENT` è un nodo con un identificatore proprio, definito nella sezione 5.
I documenti sono conservati come provenienza mediante `source_id`, senza richiedere un ulteriore tag estratto.

In [2]:
ENTITY_TYPES = {
    "COMPANY": {
        "definition": "Impresa, banca o società di gestione identificata nel testo.",
        "boundary": "Fondo ammesso solo come organizzazione investitrice, mai come quota/ETF. Niente ticker o parent company completati dal modello.",
        "source": "[F1] COMP; rinomina.",
    },
    "ORGANIZATION": {
        "definition": "Governo, regolatore, banca centrale, università o altra organizzazione non aziendale.",
        "boundary": "Il luogo è LOCATION; l'istituzione che agisce è ORGANIZATION. Ruolo solo se esplicito.",
        "source": "[F1] ORG + ORG/GOV + ORG/REG; accorpamento.",
    },
    "PRODUCT": {
        "definition": "Bene o servizio identificabile, venduto, fornito o prodotto.",
        "boundary": "Include commodity come beni scambiati (convenzione locale); una tecnologia generale è CONCEPT.",
        "source": "[F1] PRODUCT; adattamento esplicito per commodity.",
    },
    "CONCEPT": {
        "definition": "Tecnologia, processo o concetto economico/politico espresso nel testo.",
        "boundary": "Politiche nominate qui per convenzione locale; un provvedimento concreto è anche un EVENT.",
        "source": "[F1] CONCEPT; nessuna lista preventiva di temi.",
    },
    "SECTOR": {
        "definition": "Settore, attività economica o mercato di prodotti esplicitamente nominato.",
        "boundary": "Non assegnare il settore di un'azienda tramite conoscenza esterna.",
        "source": "[F1] SECTOR.",
    },
    "LOCATION": {
        "definition": "Paese, città o regione geografica.",
        "boundary": "Non usare per impianti; non convertire automaticamente US in governo statunitense.",
        "source": "[F1] GPE; rinomina.",
    },
    "INDICATOR": {
        "definition": "Variabile economica esplicita: capacità, domanda, ricavi, costi, utili, tassi.",
        "boundary": "Il valore è un attributo. owner_id se esplicito: ricavi di A e di B sono distinti. owner_id è un campo, non un ruolo: il proprietario entra in ACTOR/TARGET solo se il testo lo descrive come partecipante.",
        "source": "[F1] ECON IND + [F2] Indicator (fig. 10); estensione agli indicatori aziendali.",
    },
}
display(pd.DataFrame.from_dict(ENTITY_TYPES, orient="index").rename_axis("tag"))

,definition,boundary,source
tag,,,
COMPANY,"Impresa, banca o società di gestione identificata nel testo.","Fondo ammesso solo come organizzazione investitrice, mai come quota/ETF. Niente ticker o parent company completati dal modello.",[F1] COMP; rinomina.
ORGANIZATION,"Governo, regolatore, banca centrale, università o altra organizzazione non aziendale.",Il luogo è LOCATION; l'istituzione che agisce è ORGANIZATION. Ruolo solo se esplicito.,[F1] ORG + ORG/GOV + ORG/REG; accorpamento.
PRODUCT,"Bene o servizio identificabile, venduto, fornito o prodotto.",Include commodity come beni scambiati (convenzione locale); una tecnologia generale è CONCEPT.,[F1] PRODUCT; adattamento esplicito per commodity.
CONCEPT,"Tecnologia, processo o concetto economico/politico espresso nel testo.",Politiche nominate qui per convenzione locale; un provvedimento concreto è anche un EVENT.,[F1] CONCEPT; nessuna lista preventiva di temi.
SECTOR,"Settore, attività economica o mercato di prodotti esplicitamente nominato.",Non assegnare il settore di un'azienda tramite conoscenza esterna.,[F1] SECTOR.
LOCATION,"Paese, città o regione geografica.",Non usare per impianti; non convertire automaticamente US in governo statunitense.,[F1] GPE; rinomina.
INDICATOR,"Variabile economica esplicita: capacità, domanda, ricavi, costi, utili, tassi.","Il valore è un attributo. owner_id se esplicito: ricavi di A e di B sono distinti. owner_id è un campo, non un ruolo: il proprietario entra in ACTOR/TARGET solo se il testo lo descrive come partecipante.",[F1] ECON IND + [F2] Indicator (fig. 10); estensione agli indicatori aziendali.


## 3. Driver degli eventi

Le otto categorie vengono da [F2], A.2.3. I codici sono normalizzati in maiuscolo;
`paper_label` mantiene il nome originale. Sono driver generali, non etichette di temi investibili.

Per ogni evento annotiamo un solo driver principale quando è identificabile; altrimenti `null`.
Le categorie possono sovrapporsi: un investimento in automazione è `STRATEGIC_ACTION` se il fatto centrale
è l'investimento; una riduzione di costi già descritta è `EFFICIENCY_COST`.
Due fatti distinti possono generare due eventi con la stessa fonte.

In [3]:
DRIVERS = {
    "SUPPLY": {"paper_label": "Supply", "scope": "Offerta, capacità produttiva, disponibilità di input."},
    "DEMAND": {"paper_label": "Demand", "scope": "Domanda, ordini, vendite o consumi come volumi."},
    "REVENUE": {"paper_label": "Revenue", "scope": "Ricavi, reddito, utili e risultati economici."},
    "EFFICIENCY_COST": {"paper_label": "Efficiency Cost", "scope": "Costi, produttività, automazione ed efficienza."},
    "STRATEGIC_ACTION": {"paper_label": "Strategic Action", "scope": "Investimenti strategici, M&A, espansione e scorpori."},
    "TECHNOLOGY_INNOVATION": {"paper_label": "Technology Innovation", "scope": "Nuovi prodotti, innovazione e adozione tecnologica."},
    "POLICY_REGULATION": {"paper_label": "Policy Regulation", "scope": "Regole, autorizzazioni, restrizioni e interventi pubblici."},
    "MACRO": {"paper_label": "Macro", "scope": "Tassi, ciclo economico, PIL e shock macroeconomici."},
}
# Gli ambiti sono istruzioni operative nostre; le otto etichette provengono da FinKario.
display(pd.DataFrame.from_dict(DRIVERS, orient="index").rename_axis("driver"))

,paper_label,scope
driver,,
SUPPLY,Supply,"Offerta, capacità produttiva, disponibilità di input."
DEMAND,Demand,"Domanda, ordini, vendite o consumi come volumi."
REVENUE,Revenue,"Ricavi, reddito, utili e risultati economici."
EFFICIENCY_COST,Efficiency Cost,"Costi, produttività, automazione ed efficienza."
STRATEGIC_ACTION,Strategic Action,"Investimenti strategici, M&A, espansione e scorpori."
TECHNOLOGY_INNOVATION,Technology Innovation,"Nuovi prodotti, innovazione e adozione tecnologica."
POLICY_REGULATION,Policy Regulation,"Regole, autorizzazioni, restrizioni e interventi pubblici."
MACRO,Macro,"Tassi, ciclo economico, PIL e shock macroeconomici."


## 4. Relazioni economiche

Un arco è una **affermazione attribuita a una fonte**, con soggetto, predicato e oggetto.
Può essere pianificata o incerta: la modalità fa parte del record.
La sola presenza di due entità nella headline non autorizza un arco.

Da [F1] vengono Operate In (esteso in `ACTIVE_IN`), Produce e Introduce (accorpati in `OFFERS`), Invests In
e le tre relazioni di impatto (accorpate in `IMPACTS`). `SUPPLIES_TO` è un nostro adattamento del sottotipo
*is provided by* del driver Supply di [F2]; `ACQUIRES` e `PARTNERS_WITH` riprendono la linea guida di 0.42.
Raise e Decrease di [F1] non sono predicati separati dalla v0.2: le variazioni quantitative vengono
rappresentate mediante eventi per ridurre il vocabolario delle relazioni. Gli indicatori condivisi
possono collegare più aziende; la direzione del cambiamento resta, per ora, nella descrizione dell'evento.
La presenza di più predicati in questa lista non obbliga a usarli: `claims=[]` è un risultato valido.

In [4]:
ACTORS = ["COMPANY", "ORGANIZATION"]
NODE_TYPES = list(ENTITY_TYPES) + ["EVENT"]
RELATIONS = {
    "ACTIVE_IN": {
        "subject_types": ["COMPANY"],
        "object_types": ["SECTOR", "CONCEPT", "PRODUCT", "LOCATION"],
        "definition": "Il testo qualifica l'attività o l'esposizione dell'azienda, in qualunque posizione: 'il produttore di chip Cobalto', 'Cobalto, che fa chip'. Se la frase afferma un singolo lancio o una produzione è OFFERS, senza duplicarla. Solo parole del testo; la nazionalità non basta per LOCATION.",
        "source": "[F1] Operate In; estensione nostra dal mercato geografico a settori, concetti e prodotti.",
    },
    "OFFERS": {
        "subject_types": ACTORS,
        "object_types": ["PRODUCT", "CONCEPT"],
        "definition": "Il testo riporta che il soggetto lancia, produce o eroga un bene, servizio, tecnologia o politica.",
        "source": "[F1] Produce + Introduce; accorpamento.",
    },
    "INVESTS_IN": {
        "subject_types": ACTORS,
        "object_types": ["COMPANY", "PRODUCT", "CONCEPT", "SECTOR", "LOCATION"],
        "definition": "Quota di minoranza in una COMPANY, oppure investimento dichiarato in prodotto, concetto, settore o luogo. Il controllo è ACQUIRES.",
        "source": "[F1] Invests In; adattamento locale dei tipi ammessi; verso COMPANY confini della linea guida 0.42; strumenti esclusi.",
    },
    "ACQUIRES": {
        "subject_types": ["COMPANY"],
        "object_types": ["COMPANY"],
        "definition": "Compra, prende il controllo o offre di comprare una società o un ramo nominato. Deal ritirato, respinto o sospeso: nessun arco.",
        "source": "Linea guida 0.42 (gold/guideline.txt), acquires.",
    },
    "PARTNERS_WITH": {
        "subject_types": ["COMPANY"],
        "object_types": ["COMPANY"],
        "definition": "Accordo, alleanza, JV, licenza o collaborazione che non è acquisizione né investimento. Simmetrica: l'ordine non conta.",
        "source": "Linea guida 0.42 (gold/guideline.txt), partners with.",
    },
    "SUPPLIES_TO": {
        "subject_types": ["COMPANY"],
        "object_types": ACTORS,
        "definition": "Fornitore → cliente, anche un contratto vinto. Conservare product_id se il bene/servizio è nominato.",
        "source": "[F2] Supply → is provided by; adattamento e inversione nostri. Fra aziende coincide con supplies di 0.42.",
    },
    "IMPACTS": {
        "subject_types": ["CONCEPT", "PRODUCT", "EVENT", "ORGANIZATION", "INDICATOR"],
        "object_types": ["COMPANY", "SECTOR", "PRODUCT", "INDICATOR"],
        "definition": "Effetto affermato con un marcatore causale ('grazie a', 'a causa di', 'on', 'due to') o un verbo di effetto ('lifts', 'hits', 'boosts', 'weighs on'); 'dopo'/'after' e 'as' indicano solo successione o simultaneità e da soli non bastano. Vale anche per un rialzo di Borsa o un rating che citano la causa (il rating resta opinione del broker). impact_polarity può essere null.",
        "source": "[F1] Impact + Positive Impact On + Negative Impact On; accorpamento e restrizione dei tipi.",
    },
}
relation_table = pd.DataFrame.from_dict(RELATIONS, orient="index")
for column in ["subject_types", "object_types"]:
    relation_table[column] = relation_table[column].map(lambda values: ", ".join(values))
display(relation_table.rename_axis("predicate"))

,subject_types,object_types,definition,source
predicate,,,,
ACTIVE_IN,COMPANY,"SECTOR, CONCEPT, PRODUCT, LOCATION","Il testo qualifica l'attività o l'esposizione dell'azienda, in qualunque posizione: 'il produttore di chip Cobalto', 'Cobalto, che fa chip'. Se la frase afferma un singolo lancio o una produzione è OFFERS, senza duplicarla. Solo parole del testo; la nazionalità non basta per LOCATION.","[F1] Operate In; estensione nostra dal mercato geografico a settori, concetti e prodotti."
OFFERS,"COMPANY, ORGANIZATION","PRODUCT, CONCEPT","Il testo riporta che il soggetto lancia, produce o eroga un bene, servizio, tecnologia o politica.",[F1] Produce + Introduce; accorpamento.
INVESTS_IN,"COMPANY, ORGANIZATION","COMPANY, PRODUCT, CONCEPT, SECTOR, LOCATION","Quota di minoranza in una COMPANY, oppure investimento dichiarato in prodotto, concetto, settore o luogo. Il controllo è ACQUIRES.",[F1] Invests In; adattamento locale dei tipi ammessi; verso COMPANY confini della linea guida 0.42; strumenti esclusi.
ACQUIRES,COMPANY,COMPANY,"Compra, prende il controllo o offre di comprare una società o un ramo nominato. Deal ritirato, respinto o sospeso: nessun arco.","Linea guida 0.42 (gold/guideline.txt), acquires."
PARTNERS_WITH,COMPANY,COMPANY,"Accordo, alleanza, JV, licenza o collaborazione che non è acquisizione né investimento. Simmetrica: l'ordine non conta.","Linea guida 0.42 (gold/guideline.txt), partners with."
SUPPLIES_TO,COMPANY,"COMPANY, ORGANIZATION","Fornitore → cliente, anche un contratto vinto. Conservare product_id se il bene/servizio è nominato.",[F2] Supply → is provided by; adattamento e inversione nostri. Fra aziende coincide con supplies di 0.42.
IMPACTS,"CONCEPT, PRODUCT, EVENT, ORGANIZATION, INDICATOR","COMPANY, SECTOR, PRODUCT, INDICATOR","Effetto affermato con un marcatore causale ('grazie a', 'a causa di', 'on', 'due to') o un verbo di effetto ('lifts', 'hits', 'boosts', 'weighs on'); 'dopo'/'after' e 'as' indicano solo successione o simultaneità e da soli non bastano. Vale anche per un rialzo di Borsa o un rating che citano la causa (il rating resta opinione del broker). impact_polarity può essere null.",[F1] Impact + Positive Impact On + Negative Impact On; accorpamento e restrizione dei tipi.


### Confini che evitano archi inventati

- **Direzione:** `SUPPLIES_TO` va dal fornitore al cliente; `INVESTS_IN` dall'investitore alla destinazione;
  `ACQUIRES` dal compratore alla società acquisita. `PARTNERS_WITH` è simmetrica.
- **Quantità:** una variazione è un evento con target INDICATOR, per esempio *capacità produttiva di Alba*.
  Se leggiamo solo “I ricavi di Alba crescono”, annotiamo l'evento e l'indicatore; non inventiamo un soggetto
  che abbia causato la crescita. Se la causa è scritta, è `IMPACTS`.
- **Impatto:** `IMPACTS` richiede un effetto espresso nel testo. La polarità è `positive`, `negative`
  oppure `null`; un aumento numerico non equivale a un beneficio. Le opinioni riportate restano
  opinioni attribuite e non diventano causalità dimostrata.
- **Annunci:** “intende investire”, “potrebbe lanciare” e “smentisce” mantengono rispettivamente
  modalità pianificata, incerta o negata. La cancellazione è un evento e non prova una relazione attiva.
- **Identità:** preserviamo le menzioni e assegniamo ID locali al documento. Entity linking e alias
  storici sono un passaggio successivo. Nessun ticker viene completato dal modello.
- **Prove:** ogni evento e claim conserva una citazione esatta della headline. Una citazione presente
  non garantisce da sola che l'interpretazione sia corretta: servirà annotazione indipendente.
- **Nessun completamento:** una controparte ignota rimane ignota; liste vuote e `null` sono consentiti.
  Un evento con una sola azienda può essere utile senza una relazione azienda–azienda.

**Predicati rinviati.** `Has`, `Control`, `Relate To`, `Is Member Of` e `Participates In` di [F1]
non entrano nel nucleo iniziale; `Announce` diventa parte della descrizione/modalità dell'evento;
`Raise` e `Decrease` sono descritti dall'evento. `SELLS`, `USES_TECHNOLOGY` ed esposizioni ai ricavi potranno
essere aggiunti se un campione annotato ne mostra la necessità. Nel frattempo cessioni e adozioni restano eventi
descritti con partecipanti e driver, senza inventare un predicato economico.

## 5. Eventi, ruoli e formato minimo

`EVENT` rappresenta un fatto o cambiamento descritto dalla fonte; ha `description`, `trigger`,
`driver`, `actor_ids`, `target_ids`, `modality`, `event_time` ed `evidence`.

Per costruire il grafo, `actor_ids` produce archi **EVENT —ACTOR→ entità** e `target_ids`
produce archi **EVENT —TARGET→ entità**. Qui ACTOR e TARGET sono **nomi di archi**, non nodi.
ACTOR identifica chi compie o pianifica l'azione; TARGET il suo oggetto o destinatario.
In una fornitura: ACTOR=fornitore, TARGET=cliente e prodotto. Un attore può essere assente.
I ruoli economici specifici sono conservati nei claim: per una fornitura, soggetto=fornitore,
oggetto=cliente, `product_id`=prodotto.

Questa reificazione e i due nomi degli archi sono **convenzioni nostre**. [F2], figura 10, conserva
subject/object come descrizioni testuali, entità, timeframe e reasoning: non definisce questi ID o ruoli.
Non copiamo il campo reasoning nel grafo dei fatti; eventuali inferenze avranno una raccolta distinta.

Un claim può avere `event_id=null` per un descrittore (`ACTIVE_IN`) o un nesso causale (`IMPACTS`).
Se evento e claim descrivono lo stesso fatto, contano come **una sola evidenza**, identificata dalla fonte.

In [5]:
STRUCTURAL_RELATIONS = {
    "ACTOR": {"subject_types": ["EVENT"], "object_types": ACTORS},
    "TARGET": {"subject_types": ["EVENT"], "object_types": list(ENTITY_TYPES)},
}
MODALITIES = ["reported", "planned", "forecast", "uncertain", "negated"]
# null indica modalità non determinabile. "reported" non significa verificato indipendentemente.
IMPACT_POLARITIES = ["positive", "negative"]

SCHEMA = {
    "version": SCHEMA_VERSION,
    "entity_types": list(ENTITY_TYPES),
    "event_node_type": "EVENT",
    "event_drivers": list(DRIVERS),
    "economic_relations": {
        name: {key: record[key] for key in ["subject_types", "object_types"]}
        for name, record in RELATIONS.items()
    },
    "structural_relations": STRUCTURAL_RELATIONS,
    "modalities": MODALITIES + [None],
    "impact_polarities": IMPACT_POLARITIES + [None],
}
display(pd.DataFrame([
    {"elemento": "Tipi di entità", "numero": len(ENTITY_TYPES)},
    {"elemento": "Tipo nodo evento", "numero": 1},
    {"elemento": "Driver economici", "numero": len(DRIVERS)},
    {"elemento": "Relazioni economiche", "numero": len(RELATIONS)},
    {"elemento": "Relazioni strutturali", "numero": len(STRUCTURAL_RELATIONS)},
]))

,elemento,numero
0,Tipi di entità,7
1,Tipo nodo evento,1
2,Driver economici,8
3,Relazioni economiche,7
4,Relazioni strutturali,2


### Provenienza e point-in-time

| Campo | Significato |
|---|---|
| `source_id`, `text` | Documento e testo originali; ogni claim contiene inoltre `evidence` |
| `published_at` | Timestamp della pubblicazione della versione utilizzata |
| `available_at` | Primo momento di disponibilità nel feed/processo, includendo il ritardo applicabile |
| `event_time` | Data/periodo del fatto se esplicito; altrimenti `null`, senza copiarvi automaticamente la data della news |
| `schema_version` | Versione delle definizioni di annotazione |

Nel backtest il filtro è **`available_at <= decision_time`**. Se la disponibilità storica è ignota,
il record non può essere certificato PIT; eventuali approssimazioni vanno dichiarate.
Conservare versioni delle fonti, correzioni e cancellazioni, senza riscrivere il passato.
ID globali, alias, universo investibile e dati aggiuntivi devono essere risolti alla stessa data.

A livello di esecuzione si registreranno anche versione del modello/prompt ed `extracted_at`.
Quest'ultimo è il momento dell'elaborazione, non la disponibilità storica della fonte.
La data della notizia e un modello datato non controllano, da soli, tutte le vie di leakage.

`confidence` non è obbligatorio nel nucleo: un numero autoassegnato dal modello non è una probabilità
calibrata. Prima di usarlo come soglia serve verificarlo su annotazioni indipendenti.

## 6. Un record completo

Headline **sintetica**: “Alba pianifica un aumento del 20% della capacità produttiva di batterie.”
Annotiamo un aumento **pianificato** di capacità; non un aumento già avvenuto, né un beneficio sui profitti.
Dalla versione 0.2 non c'è un claim: l'aumento è l'evento (target *capacità produttiva*), e il legame
Alba–batterie passa da ACTOR e TARGET.
La data di disponibilità qui è costruita per l'esempio, non misurata su un feed.

In [6]:
text = "Alba pianifica un aumento del 20% della capacità produttiva di batterie."
EXAMPLE = {
    "schema_version": SCHEMA_VERSION,
    "source_id": "synthetic:001",
    "published_at": "2024-06-03T08:00:00Z",
    "available_at": "2024-06-03T08:01:00Z",
    "text": text,
    "entities": [
        {"id": "d1:e1", "type": "COMPANY", "mention": "Alba"},
        {"id": "d1:e2", "type": "PRODUCT", "mention": "batterie"},
        {
            "id": "d1:e3", "type": "INDICATOR",
            "mention": "capacità produttiva", "owner_id": "d1:e1",
        },
    ],
    "events": [{
        "id": "d1:v1",
        "type": "EVENT",
        "description": "Aumento pianificato del 20% della capacità produttiva di batterie di Alba.",
        "trigger": "pianifica",
        "driver": "SUPPLY",
        "actor_ids": ["d1:e1"],
        "target_ids": ["d1:e3", "d1:e2"],
        "modality": "planned",
        "event_time": None,
        "evidence": text,
    }],
    "claims": [],
}
print(json.dumps(EXAMPLE, ensure_ascii=False, indent=2))

{
  "schema_version": "0.3",
  "source_id": "synthetic:001",
  "published_at": "2024-06-03T08:00:00Z",
  "available_at": "2024-06-03T08:01:00Z",
  "text": "Alba pianifica un aumento del 20% della capacità produttiva di batterie.",
  "entities": [
    {
      "id": "d1:e1",
      "type": "COMPANY",
      "mention": "Alba"
    },
    {
      "id": "d1:e2",
      "type": "PRODUCT",
      "mention": "batterie"
    },
    {
      "id": "d1:e3",
      "type": "INDICATOR",
      "mention": "capacità produttiva",
      "owner_id": "d1:e1"
    }
  ],
  "events": [
    {
      "id": "d1:v1",
      "type": "EVENT",
      "description": "Aumento pianificato del 20% della capacità produttiva di batterie di Alba.",
      "trigger": "pianifica",
      "driver": "SUPPLY",
      "actor_ids": [
        "d1:e1"
      ],
      "target_ids": [
        "d1:e3",
        "d1:e2"
      ],
      "modality": "planned",
      "event_time": null,
      "evidence": "Alba pianifica un aumento del 20% della capacità 

### Casi di confine per l'annotazione

Tutti gli esempi sono inventati; servono a chiarire le definizioni, non a misurare un modello.

| Headline | Annotazione minima | Cosa non aggiungere |
|---|---|---|
| Borea fornisce sistemi di raffreddamento ad Alba. | `Borea —SUPPLIES_TO→ Alba`, prodotto esplicito; evento `SUPPLY` | Non dedurre che Borea produca il sistema o che il cliente sia il suo principale mercato |
| Cobalto presenta il modello linguistico Lira. | `Cobalto —OFFERS→ Lira`; evento `TECHNOLOGY_INNOVATION` | Non aggiungere clienti, chip o vincitori di mercato |
| Alba adotta una nuova tecnologia di riciclo. | EVENT, attore Alba, target CONCEPT, driver `TECHNOLOGY_INNOVATION`; `claims=[]` | Non inventare `INVESTS_IN` o una controparte |
| I ricavi di Alba aumentano del 10%. | EVENT `REVENUE`, target INDICATOR con owner Alba; `claims=[]` | Non affermare che Alba o una tecnologia abbiano causato l'aumento |
| L'Autorità vieta la vendita del prodotto Lira. | EVENT `POLICY_REGULATION`, attore ORGANIZATION, target PRODUCT | Non dedurre automaticamente un impatto negativo sugli utili |
| Alba smentisce l'acquisizione di Borea. | EVENT `STRATEGIC_ACTION`, modalità `negated`; `claims=[]` | Non creare un'acquisizione attiva |
| Alba e Borea in rialzo in Borsa. | Due COMPANY; `events=[]`, `claims=[]` nel perimetro iniziale | Non trasformare una co-menzione in partnership o fornitura |

Un filtro successivo decide quali modalità sono ammesse nel segnale: una dichiarazione pianificata può
essere informativa al momento della pubblicazione, purché non venga trattata come fatto completato.

## 7. Controllo locale del formato

La cella verifica solo coerenza dei riferimenti, tipi ammessi ed evidenza presente nel record sintetico.
Non valuta verità, causalità, qualità dell'estrazione o assenza di leakage.

In [7]:
entities = {entity["id"]: entity for entity in EXAMPLE["entities"]}
events = {event["id"]: event for event in EXAMPLE["events"]}
nodes = {**entities, **events}
assert len(nodes) == len(EXAMPLE["entities"]) + len(EXAMPLE["events"])
assert EXAMPLE["schema_version"] == SCHEMA["version"]

for entity in entities.values():
    assert entity["type"] in SCHEMA["entity_types"]
    assert entity["mention"] in EXAMPLE["text"]
    if "owner_id" in entity:
        assert entity["owner_id"] in entities

for event in events.values():
    assert event["type"] == SCHEMA["event_node_type"]
    assert event["driver"] is None or event["driver"] in SCHEMA["event_drivers"]
    assert event["modality"] in SCHEMA["modalities"]
    assert event["trigger"] in EXAMPLE["text"]
    assert event["evidence"] and event["evidence"] in EXAMPLE["text"]
    assert all(entities[node_id]["type"] in ACTORS for node_id in event["actor_ids"])
    assert all(node_id in entities for node_id in event["target_ids"])

for claim in EXAMPLE["claims"]:
    rule = SCHEMA["economic_relations"][claim["predicate"]]
    assert nodes[claim["subject_id"]]["type"] in rule["subject_types"]
    assert nodes[claim["object_id"]]["type"] in rule["object_types"]
    assert claim["event_id"] is None or claim["event_id"] in events
    assert claim["modality"] in SCHEMA["modalities"]
    assert claim["impact_polarity"] in SCHEMA["impact_polarities"]
    assert claim["predicate"] == "IMPACTS" or claim["impact_polarity"] is None
    assert claim["evidence"] and claim["evidence"] in EXAMPLE["text"]
    if claim.get("product_id") is not None:
        assert entities[claim["product_id"]]["type"] == "PRODUCT"

print(f"Record sintetico coerente con lo schema {SCHEMA_VERSION}.")

Record sintetico coerente con lo schema 0.3.


## 8. Collegamento con 0.42 e criterio per ampliare lo schema

[0.42](0.42-pit-datedgpt-gpt-leakage.ipynb) valuta una relazione tra due attori economici nominati.
Questa proposta ammette più eventi e claim per headline, anche con una sola azienda.
Il gold di 0.42 resta utilizzabile solo per `ACQUIRES`, `PARTNERS_WITH`, `INVESTS_IN` e `SUPPLIES_TO` fra aziende,
che ne riprendono le definizioni; il resto richiede riannotazione. Le definizioni 0.2–0.3 di `ACTIVE_IN` e `IMPACTS`
sono state scritte guardando i 150 titoli 2024 di 0.42, dev e test: per queste il test va estratto da un campione nuovo.

Il primo esperimento dovrebbe usare un campione cronologico di headline, includendo negativi e testi
senza i trigger societari di 0.42. Si congelano schema e linee guida sul development set; il test resta
separato, con revisione umana indipendente. Si misurano precisione, richiamo e direzione dei claim,
qualità dei driver, modalità e rispetto dell'evidenza. I casi non coperti vengono registrati, non forzati
nel predicato più vicino.

Per la discovery si osserveranno nuove connessioni, crescita del supporto documentale indipendente,
persistenza e diffusione fra aziende/settori. La co-occorrenza può proporre candidati; il grafo economico
ne documenta i collegamenti. Un nuovo predicato entra soltanto se un errore ricorrente mostra che i ruoli
evento e i sette predicati non conservano informazione utile. Non sono richiesti GNN o GraphRAG per
valutare questo primo schema.

## 9. Stime di scala: tempi e costi indicativi

I corpus locali contengono **22.761.199** headline (`news_corpus.parquet`) e **115.473.962**
headline (`news_corpus_press.parquet`). Con velocità sostenuta `r`, il tempo è `N / (r × 86.400)` giorni:

| News complete al secondo | Corpus da 22,8 milioni | Corpus da 115,5 milioni |
|---|---:|---:|
| 5 | 52,7 giorni | 267,3 giorni |
| 20 | 13,2 giorni | 66,8 giorni |
| 100 | 2,6 giorni | 13,4 giorni |

**Sono scenari, non benchmark A100.** Per finire in 14 giorni, lavorando 24/7, servono circa
**19 news/s** oppure **96 news/s**, rispettivamente. Un'A100 da 80 GB consente batch più grandi
rispetto a 40 GB, ma la memoria non determina da sola il throughput. Misurare separatamente
GLiNER2.5 e Qwen, includendo validazione, salvataggio e tentativi aggiuntivi; la velocità di una
singola risposta non basta. Per l'A100 da 40 GB, Qwen3-8B è il primo candidato locale da provare.

Prezzi API verificati il **29 settembre 2026**: ipotizzando **1.500 token di input + 300 token
fatturabili di output per headline**, modalità Batch, senza cache, retry o altri costi:

| Modello | Costo stimato per 1 milione di headline, USD |
|---|---:|
| GPT-4o | 3.375 |
| GPT-5.5 | 8.250 |
| Gemini 2.5 Flash | 600 |
| Gemini 2.5 Pro | 2.438 |

Lunghezze non ancora misurate sul nuovo schema; conteggiare anche i token di ragionamento fatturabili.
Fonti: [GPT-4o](https://developers.openai.com/api/docs/models/gpt-4o),
[GPT-5.5](https://developers.openai.com/api/docs/models/gpt-5.5),
[sconto Batch OpenAI](https://developers.openai.com/api/docs/guides/batch),
[prezzi Gemini](https://ai.google.dev/gemini-api/docs/pricing).

Il primo confronto di accuratezza e leakage è pianificato in
[0.44 — Estrazione e leakage temporale](0.44-ontology-extraction-and-temporal-leakage.ipynb).
